# MADLAD-400-3B 翻译 · Colab GPU 跑 + 本地存档

**目标**：用 Colab 免费 T4 GPU 把开源翻译模型 MADLAD-400-3B 跑起来，验证它翻你社区文本的质量；顺手把量化后的小模型（~3G）打包下载回本地存档。

**策略**：
- 重活（下 12G 本体 + 量化）在 Colab 快网上做，本地只留 3G 成品
- T4 有 16G 显存，MADLAD-3B（int8 ~3G）在 GPU 上跑得飞快
- 这是学习笔记，随时在 markdown cell 里记你自己的观察

> 用法：菜单 **代码执行程序 → 更改运行时类型 → 硬件加速器选 T4 GPU**，然后从上往下逐格运行。

## 0. 确认拿到了 GPU

下面应该打印出一张 Tesla T4（16G 显存）。如果报错说没有 GPU，回去改运行时类型。

In [1]:
!nvidia-smi

Fri Jul 10 06:56:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. 装依赖

Colab 自带 `torch`，我们只补三样：
- **ctranslate2**：高效推理引擎，负责 int8 量化 + 跑模型（比原生快数倍）
- **transformers**：只用它的分词器（把文字切成模型认识的 token）
- **sentencepiece**：MADLAD 分词器底层需要它

In [7]:
# 两个包都锁旧版，且必须是同时代的组合（2024 年版），原因：
# - transformers 锁 4.44.2：MADLAD 检查点里存了两份不一致的共享权重，
#   新版 transformers 不再强制 tie，转换时输出层用错权重 → 译文变复读机
# - ctranslate2 锁 4.5.0：新版 ct2 按新版 transformers 接口传参（dtype），
#   旧版 transformers 只认 torch_dtype，隔代搭配直接 TypeError
!pip install -q "ctranslate2==4.5.0" "transformers==4.44.2" sentencepiece

In [10]:
!pip install -q --force-reinstall "ctranslate2==4.5.0" "transformers==4.44.2" sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 74.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/801.3 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import ctranslate2, transformers
print("ctranslate2:", ctranslate2.__version__, "| transformers:", transformers.__version__)


ctranslate2: 4.5.0 | transformers: 4.44.2


## 2. 下载 F32 本体 + 量化成 CT2 int8

这一步做两件事：
1. 从 HuggingFace 下载 MADLAD-3B 的全精度权重（约 12G，Colab 网快，一两分钟）
2. 用 `ct2-transformers-converter` 把它量化成 **int8**（12G → ~3G），产出一个自包含文件夹 `madlad400-3b-ct2-int8/`

**为什么量化**：int8 用 1 字节存一个权重（原本 4 字节），体积和显存直接砍到 ~1/4，速度还更快，质量几乎无损。

> 用纯 `int8`（不是 int8_float16）：这样这份成品在 GPU 和 CPU 上都能跑，本地存档回去用 CPU 也顺。转换是纯 CPU 过程，耐心等几分钟。

In [2]:
# 用 Python API 转换（CLI 暴露不了 load_as_float16 选项）
# - load_as_float16：以 fp16 读权重，峰值内存从 ~12G 减半到 ~6G（Colab 只有 12.67G RAM，fp32 会被 OOM 杀掉）
# - low_cpu_mem_usage：边读边转，进一步压低峰值
# 反正最终要量化成 int8，fp32→fp16 这步对质量无感
import ctranslate2.converters

conv = ctranslate2.converters.TransformersConverter(
    "jbochi/madlad400-3b-mt",
    load_as_float16=True,
    low_cpu_mem_usage=True,
)
conv.convert("madlad400-3b-ct2-int8", quantization="int8", force=True)
print("转换完成 ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


转换完成 ✅


## 3. 把 int8 模型加载到 GPU，定义翻译函数

**MADLAD 的用法**：在原文前加一个 `<2xx>` 前缀指定目标语言（`<2en>` 英、`<2ja>` 日、`<2fr>` 法…），源语言模型自动识别。所以一个模型能「任意语言 → 任意语言」。

In [ ]:
import ctranslate2, transformers

# 加载 int8 模型到 GPU（T4 16G 显存，3B 绰绰有余）
translator = ctranslate2.Translator("madlad400-3b-ct2-int8", device="cuda", compute_type="int8")
tokenizer = transformers.AutoTokenizer.from_pretrained("jbochi/madlad400-3b-mt")

def translate(text: str, target: str) -> str:
    """把 text 翻成 target（en/zh/ja/fr...）。MADLAD 用 <2xx> 前缀指定目标语言。
    注意：语言代码必须用模型词表里的（中文是 zh，不是 zh-cn）；长文按段落切开逐段调用。"""
    prompt = f"<2{target}> {text}"
    src = tokenizer.convert_ids_to_tokens(tokenizer.encode(prompt))
    res = translator.translate_batch([src], beam_size=1, max_decoding_length=512)
    ids = tokenizer.convert_tokens_to_ids(res[0].hypotheses[0])
    return tokenizer.decode(ids, skip_special_tokens=True)

print("模型就绪，测一句：", translate("你好，欢迎来到我们的社区", "en"))

## 4. 翻你自己的文本 ⭐

**把下面的 `samples` 换成你社区里真实的句子**（口语、俚语、缩写都来），才测得出这模型对你到底够不够用。这是整个 notebook 最关键的一步——质量判断只有用你的真实数据才算数。

In [10]:
# ⭐ 长文翻译的正确姿势，两个要点：
# 1. 语言代码用 zh（不是 zh-cn）——MADLAD 词表里只有 <2zh> 这个 token，zh-cn 不被识别
# 2. MADLAD 是句子级翻译模型，整篇塞进去会崩；按段落切开逐段翻
long_text = """The Korean "Supreme God" Mobile Game: A Masterclass in the Psychology of Pay-to-Win
On June 18th, the Korean server released an idle mobile game, Sol:enchant, with a mechanism so absurd I was stunned for a few seconds after seeing it: each server selects a "god," and the player who spends the most money across the entire server is directly crowned "Supreme God." (Gamersky (https://www.gamersky.com/news/))
Just look at how ruthless this design is. Ordinary pay-to-win games are about "spending money to become stronger," and that strength is private and hidden. This game directly turns spending money into a server-wide public competition for status—you're not buying equipment, you're buying a god-like position where "the whole server kneels before you." This instantly upgrades the competition among whale players from "who is stronger" to "who is a god," ripping off the ceiling for spending money.
I find its monetization method disgusting, but I have to admit it's a shrewd design that directly monetizes social status. Find a target that users are willing to spend madly on, and then make it quantifiable and boastful. Its target is vanity, disgusting, but effective."""

# 按行切段，逐段翻译
paragraphs = [p.strip() for p in long_text.split("\n") if p.strip()]
for para in paragraphs:
    print("原文：", para)
    print("译文：", translate(para, "zh"))
    print()

原文： The Korean "Supreme God" Mobile Game: A Masterclass in the Psychology of Pay-to-Win
译文： 韩国"最高神"移动游戏:付费赢心理学大师课

原文： On June 18th, the Korean server released an idle mobile game, Sol:enchant, with a mechanism so absurd I was stunned for a few seconds after seeing it: each server selects a "god," and the player who spends the most money across the entire server is directly crowned "Supreme God." (Gamersky (https://www.gamersky.com/news/))
译文： 6月18日,韩国服务器发布了一个闲置的移动游戏,Sol:enchant,其机制如此荒谬,我在看到它后几秒钟就惊呆了:每个服务器选择一个"神 ", 花费最多的玩家在整个服务器直接加冕"最高神 " 。 (Gamersky (https : / / www.gamersky.com/news/ )

原文： Just look at how ruthless this design is. Ordinary pay-to-win games are about "spending money to become stronger," and that strength is private and hidden. This game directly turns spending money into a server-wide public competition for status—you're not buying equipment, you're buying a god-like position where "the whole server kneels before you." This instantly upgrades the competition among wh

## 5. 量一下 GPU 上的速度

跑几句测平均延迟，感受一下 T4 GPU + int8 有多快（对比你本机 CPU 的几秒/句）。

In [7]:
import time
translate("热身一下", "en")  # 预热
t = time.perf_counter()
for _ in range(10):
    translate("我们社区今晚有一场直播，欢迎来看。", "en")
dt = (time.perf_counter() - t) / 10
print(f"GPU 单句平均延迟：{dt*1000:.0f} ms")

GPU 单句平均延迟：222 ms


## 6. 打包 int8 成品，下载回本地存档

把 `madlad400-3b-ct2-int8/`（~3G）打包成 zip 下载。存到本地后：
- 以后不用再下 12G，这份成品离线可跑
- 放进本机 `ai_book/translation/`，用 `server.py`（`CT2_DEVICE=cpu`）就能在你电脑上跑
- 将来上 4090 生产，同一份直接复用

In [ ]:
!zip -r -q madlad400-3b-ct2-int8.zip madlad400-3b-ct2-int8
print("打包完成，开始下载（约 3G，视网速）...")
from google.colab import files
files.download("madlad400-3b-ct2-int8.zip")

## 小结 & 下一步

**你在这个 notebook 里学到/验证了**：下载 → 量化(int8) → GPU 加载 → 翻译 → 压测 → 打包存档，一整条自建翻译流水线。

**接着可以**：
- 在这下面加 markdown cell，记你对**译文质量**的判断（哪些好、哪些翻车）——这决定要不要上生产
- 质量满意 → 租 4090 用这份 int8 成品上常驻服务（`CT2_DEVICE=cuda`）
- 想在本机跑 → 把下载的 zip 解压进 `ai_book/translation/`，起 `server.py`

> 📝 你的笔记区（随便写）：
> - 译文质量观察：
> - 遇到的坑：
> - 下一步想试：